# v2 · 02 · Bank-style points scorecard
v1 showed graph features make **logistic regression** reliably better. Banks usually deploy logistic regression as a **points scorecard**: each feature is cut into bands, each band gets points, and the points add up to a score.

Here we build two scorecards with the same method: **without** and **with** graph features. Bins and points are learned on training customers only; the test set is used once.

**Setup.**

In [1]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))      # v2 package (adds v1 src to the path too)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from credit_kg_v2 import load_config, OUT
from credit_kg import config as C, scoring as S, modeling as M
from credit_kg_v2 import scorecard as SC
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
scfg = load_config()['scorecard']

**Load v1's feature table** (behaviour + graph features) and the fixed v1 split.

In [2]:
table = M.load_table()
tr, te = table[table.split == "train"], table[table.split == "test"]
F0 = scfg["features_without_graph"]; F1 = F0 + scfg["graph_features"]
print("without graph:", F0); print("graph features added:", scfg["graph_features"])

without graph: ['late_months', 'late_recent3', 'status_latest', 'max_status', 'coverage_median', 'util_avg', 'credit_limit', 'pay_median', 'high_util_months', 'zero_pay_months']
graph features added: ['g_markov_next_late_prob', 'g_relapses', 'g_recoveries', 'g_path_typicality', 'g_weak_cov_after_late']


**How a scorecard is built** (per feature, on training data):
1. cut into ~5 bands by quantile;
2. merge neighbouring bands until default rates move in one direction (utilization may stay U-shaped, as validated in v1);
3. turn each band into a *weight of evidence* (how much safer or riskier than average);
4. fit logistic regression on those values;
5. convert to points: 600 points = odds of 50 good : 1 bad, and every 20 points doubles the odds.

In [3]:
sc0 = SC.Scorecard(F0).fit(tr[F0], tr[C.TARGET])
sc1 = SC.Scorecard(F1).fit(tr[F1], tr[C.TARGET])
pts = sc1.points_table(); pts.to_csv(OUT / "scorecard_points_with_graph.csv", index=False)
pts

,feature,bin,customers_train,default_rate_train,woe,points
0,late_months,"(-inf, 1.0]",19468,0.149,0.484,37
1,late_months,"(1.0, inf]",4532,0.532,-1.385,30
2,late_recent3,"(-inf, 1.0]",20563,0.163,0.376,36
3,late_recent3,"(1.0, inf]",3437,0.569,-1.534,31
4,status_latest,"(-inf, 0.0]",18506,0.137,0.579,47
5,status_latest,"(0.0, 1.0]",3000,0.344,-0.612,22
6,status_latest,"(1.0, inf]",2494,0.697,-2.091,-9
7,max_status,"(-inf, 0.0]",15896,0.115,0.781,41
8,max_status,"(0.0, inf]",8104,0.429,-0.974,28
9,coverage_median,"(-inf, 0.0394]",4800,0.312,-0.469,35


**Cross-validation on training data** (5 folds; bins and points re-learned inside each fold).

In [4]:
cv = StratifiedKFold(5, shuffle=True, random_state=C.SEED)
rows = []
for k, (a, b) in enumerate(cv.split(tr, tr[C.TARGET])):
    A, B = tr.iloc[a], tr.iloc[b]
    for name, F in [("scorecard without graph", F0), ("scorecard with graph", F1)]:
        p = SC.Scorecard(F).fit(A[F], A[C.TARGET]).predict_proba(B[F])
        rows.append({"fold": k, "model": name, "roc_auc": roc_auc_score(B[C.TARGET], p), "pr_auc": average_precision_score(B[C.TARGET], p)})
cvr = pd.DataFrame(rows); cvr.groupby("model")[["roc_auc", "pr_auc"]].agg(["mean", "std"]).round(4)

roc_auc          pr_auc        
                           mean     std    mean     std
model                                                  
scorecard with graph     0.7710  0.0085  0.5304  0.0191
scorecard without graph  0.7692  0.0089  0.5271  0.0188

**Test set, used once**, next to the v1 models for reference.

In [5]:
p0, p1 = sc0.predict_proba(te[F0]), sc1.predict_proba(te[F1]); yt = te[C.TARGET].to_numpy()
boot = M.paired_bootstrap(yt, p0, p1)
v1 = pd.read_csv(C.TABLES / "model_comparison_lr_test.csv", index_col=0); v1rf = pd.read_csv(C.TABLES / "model_comparison_test.csv", index_col=0)
comp = pd.DataFrame({"ROC-AUC": [roc_auc_score(yt, p0), roc_auc_score(yt, p1), v1.loc["roc_auc", "LR-A"], v1.loc["roc_auc", "LR-B"], v1rf.loc["roc_auc", "A (baseline)"]],
                     "features": [len(F0), len(F1), 37, 46, 37]},
                    index=["v2 scorecard without graph", "v2 scorecard with graph", "v1 logistic regression A", "v1 logistic regression B", "v1 Random Forest A"])
print(f"graph effect inside the scorecard: ΔROC-AUC {boot['delta_roc_auc']:+.4f}, 95% CI [{boot['roc_auc_ci'][0]:+.4f}, {boot['roc_auc_ci'][1]:+.4f}]")
comp.round(4)

graph effect inside the scorecard: ΔROC-AUC +0.0011, 95% CI [-0.0030, +0.0053]


,ROC-AUC,features
v2 scorecard without graph,0.7563,10
v2 scorecard with graph,0.7574,15
v1 logistic regression A,0.7573,37
v1 logistic regression B,0.7638,46
v1 Random Forest A,0.7800,37


**Do more points really mean less default?** Test customers split into five equal groups by scorecard points.

In [6]:
tp = sc1.score(te[F1])
grp = pd.qcut(tp, 5, labels=["lowest 20%", "2nd", "3rd", "4th", "highest 20%"])
g = pd.DataFrame({"points_range": tp.groupby(grp).agg(lambda s: f"{s.min():.0f}-{s.max():.0f}"), "customers": tp.groupby(grp).size(),
                  "default_rate": te[C.TARGET].groupby(grp).mean().round(3)})
g.to_csv(OUT / "scorecard_default_by_points.csv"); g

,points_range,customers,default_rate
scorecard_points,,,
lowest 20%,446-511,1244,0.528
2nd,512-537,1166,0.226
3rd,538-546,1262,0.147
4th,547-555,1227,0.122
highest 20%,556-577,1101,0.064


**One customer's scorecard**: every band and its points, adding up to the total. This is the explanation a credit officer would see.

In [7]:
row = te.iloc[[0]]
ex = sc1.explain(row[F1]); print(f"customer {int(row.customer_id.iloc[0])}: total {ex.points.sum():.0f} points  (actual outcome: {'default' if row[C.TARGET].iloc[0] else 'no default'})")
ex

customer 2: total 517 points  (actual outcome: default)


,feature,value,bin,points
0,late_months,2.000000,"(1.0, inf]",30
1,late_recent3,1.000000,"(-inf, 1.0]",36
2,status_latest,-1.000000,"(-inf, 0.0]",47
3,max_status,2.000000,"(0.0, inf]",28
4,coverage_median,0.289436,"(0.198, inf]",35
5,util_avg,0.023718,"(0.018, 0.132]",43
6,credit_limit,120000.000000,"(100000.0, 180000.0]",36
7,pay_median,1000.000000,"(485.9, 1288.8]",30
8,high_util_months,0.000000,"(-inf, 2.0]",35
9,zero_pay_months,2.000000,"(1.0, inf]",31


**Save** the comparison.

In [8]:
comp.to_csv(OUT / "scorecard_comparison_test.csv"); print("saved")

saved


### Takeaways
* **A 15-feature points scorecard matches v1's 37-feature logistic regression** (test ROC-AUC 0.757 vs 0.757) while being far easier to read: every customer's score is a sum of visible points.
* **Points mean what they should:** default rate falls from 53% (lowest 20% of points) to 6% (highest 20%) on test customers.
* **Graph features inside the compact scorecard: no reliable gain** (+0.0011 ROC-AUC, 95% CI −0.0030 to +0.0053). Honest reasons:
  - banding coarsens the graph features; `g_relapses` and `g_recoveries` collapsed into a single band (so they carry no points), because enforcing a one-direction pattern merged their bins;
  - v1's gain (+0.0065) came from the full 46-feature logistic regression, where the graph features enter as exact numbers.
* **Next step:** let the graph features keep more bands (or enter as continuous terms) and re-test with the full v1 protocol (25 folds + bootstrap).